# LeetCode 3090 — Maximum Length Substring With Two Occurrences bằng Python

Link đề: https://leetcode.com/problems/maximum-length-substring-with-two-occurrences/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maximumLengthSubstring(self, s: str) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **độ dài lớn nhất** của một **chuỗi con liên tiếp** của `s` sao cho **mỗi ký tự xuất hiện tối đa 2 lần**.
2. Nắm các công cụ Python: **cắt lát** `s[i:j]`, **đếm** bằng `Counter` / `dict` / mảng `26` phần tử, `max`, và khái niệm **hai con trỏ** (`left`, `right`).
3. Đi từ cách **thử mọi chuỗi con và đếm lại** → **với mỗi điểm bắt đầu, mở rộng sang phải tới khi vi phạm** → **cửa sổ trượt (sliding window) với `Counter`** → **cửa sổ trượt với mảng đếm `26`** → **cửa sổ trượt nhảy thẳng `left` nhờ nhớ 2 vị trí gần nhất của mỗi ký tự**.
4. Hiểu mấu chốt: khi một ký tự có **số lần > 2** trong cửa sổ, **thu hẹp từ bên trái** cho tới khi hết vi phạm; trong lúc đó cửa sổ **luôn hợp lệ** tại các thời điểm ta ghi nhận đáp án.
5. Tránh các lỗi kinh điển: dùng `if` thay `while` khi thu hẹp (với cách cửa sổ co lại), quên **giảm đếm** khi bỏ ký tự bên trái, cập nhật đáp án **trước** khi thu hẹp, nhầm "tối đa 2 lần" thành "đúng 2 lần" hay "khác nhau".

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. Hãy trả về **độ dài lớn nhất** của một **chuỗi con** của `s` sao cho **mỗi ký tự xuất hiện không quá 2 lần** trong chuỗi con đó.

(Chuỗi con là dãy ký tự **liên tiếp** trong `s`.)

**Ví dụ 1**

- Input: `s = "bcbbbcba"`
- Output: `4`
- Giải thích: chuỗi con `"bcba"` (các ký tự ở chỉ số `4..7`) có độ dài `4`; `b` xuất hiện `2` lần, `c` và `a` mỗi ký tự `1` lần. Không có chuỗi con dài hơn thỏa điều kiện.

**Ví dụ 2**

- Input: `s = "aaaa"`
- Output: `2`
- Giải thích: chuỗi con dài nhất mà `a` không quá `2` lần là `"aa"`.

**Ví dụ 3 (tự dựng)**

- Input: `s = "abcabcabc"`
- Output: `6`
- Giải thích: `"abcabc"` có mỗi ký tự `2` lần; thêm một ký tự nữa thì có ký tự thành `3` lần.

**Ràng buộc:** `2 <= s.length <= 100`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
from typing import List
from collections import Counter
import random
import string

examples = [
    ("bcbbbcba", 4),
    ("aaaa", 2),
    ("abcabcabc", 6),
]
for s, expected in examples:
    print(repr(s), "-> kỳ vọng", expected)

'bcbbbcba' -> kỳ vọng 4
'aaaa' -> kỳ vọng 2
'abcabcabc' -> kỳ vọng 6


## 3. Kiến thức nền

### 3.1. Cắt lát chuỗi con: `s[i:j]`

`s[i:j]` là chuỗi con từ chỉ số `i` tới **trước** `j` (không gồm `j`); độ dài `j - i`. Với `s` dài `n`, có `n * (n + 1) / 2` chuỗi con không rỗng. Cắt lát `s[0:n]` là toàn bộ chuỗi.

In [2]:
s = "bcba"
print(s[1:3])                  # cb
print(s[0:4], s[2:3])          # bcba b
n = len(s)
print(n * (n + 1) // 2, "chuỗi con không rỗng:", [s[i:j] for i in range(n) for j in range(i + 1, n + 1)])

cb
bcba b
10 chuỗi con không rỗng: ['b', 'bc', 'bcb', 'bcba', 'c', 'cb', 'cba', 'b', 'ba', 'a']


### 3.2. Đếm bằng `Counter` và `dict`

`Counter("bcbb")` cho `{'b': 3, 'c': 1}`. `max(Counter(sub).values())` là số lần lặp nhiều nhất của một ký tự; điều kiện của đề là giá trị này `<= 2`. Với `dict` thường ta dùng `cnt[c] = cnt.get(c, 0) + 1`.

In [3]:
c = Counter("bcbb")
print(c, max(c.values()))              # Counter({'b': 3, 'c': 1}) 3
cnt = {}
for ch in "bcbb":
    cnt[ch] = cnt.get(ch, 0) + 1
print(cnt)                             # {'b': 3, 'c': 1}
print(max(Counter("bcba").values()) <= 2)     # True

Counter({'b': 3, 'c': 1}) 3
{'b': 3, 'c': 1}
True


### 3.3. Mảng đếm 26 phần tử và `ord`

Vì chuỗi chỉ có chữ thường, ta có thể đếm bằng một **list 26 số**: `count[ord(c) - ord('a')] += 1`. Cách này nhanh và không cần `dict`. `ord('a') = 97`.

In [4]:
count = [0] * 26
for ch in "bcba":
    count[ord(ch) - ord("a")] += 1
print(count[:4])               # [1, 2, 1, 0] -> a:1, b:2, c:1, d:0

[1, 2, 1, 0]


### 3.4. Hai con trỏ và cửa sổ trượt (sliding window)

**Cửa sổ** là đoạn `s[left..right]` (cả hai đầu **gồm**). Ta dịch `right` sang phải **từng bước** (thêm một ký tự), và khi cửa sổ **vi phạm** điều kiện thì dịch `left` sang phải (bỏ ký tự bên trái) cho tới khi cửa sổ lại hợp lệ. Mỗi chỉ số `left` và `right` chỉ tăng, không bao giờ lùi ⇒ tổng số bước `O(n)`.

Độ dài cửa sổ là `right - left + 1`.

In [5]:
s = "bcba"
left = 0
for right in range(len(s)):
    print(f"right={right}: cửa sổ s[{left}..{right}] = {s[left:right + 1]!r}, độ dài {right - left + 1}")

right=0: cửa sổ s[0..0] = 'b', độ dài 1
right=1: cửa sổ s[0..1] = 'bc', độ dài 2
right=2: cửa sổ s[0..2] = 'bcb', độ dài 3
right=3: cửa sổ s[0..3] = 'bcba', độ dài 4


### 3.5. Vòng lặp `while` để thu hẹp

Khi thêm ký tự mới làm một ký tự vượt `2` lần, ta có thể phải bỏ **nhiều** ký tự bên trái mới hết vi phạm (không biết trước). Vì vậy dùng `while` (lặp **cho tới khi** hết vi phạm), không dùng `if` (chỉ làm một lần). Mỗi lần bỏ ký tự `s[left]`: **giảm đếm của nó đi `1`** rồi `left += 1`.

In [6]:
cnt = Counter("bbb")              # b: 3 (vi phạm)
left = 0
s = "bbb"
while cnt["b"] > 2:
    cnt[s[left]] -= 1             # bỏ ký tự ngoài cùng bên trái
    left += 1
print(left, cnt)                  # 1 Counter({'b': 2})

1 Counter({'b': 2})


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `s = "bcbbbcba"` (chỉ số `0..7`: `b c b b b c b a`). Ta thử các cửa sổ bắt đầu từ `0`:

| bắt đầu `i` | mở rộng sang phải tới khi vi phạm | chuỗi con hợp lệ dài nhất | độ dài |
|---|---|---|---|
| 0 | `b c b b` có `b` ×3 ở chỉ số 3 ⇒ dừng trước | `bcb` | 3 |
| 1 | `c b b` rồi `b` ×3 ⇒ dừng | `cbb` | 3 |
| 2 | `b b` rồi `b` ×3 ⇒ dừng | `bb` | 2 |
| 3 | `b b` rồi `c` rồi `b` ×3 ⇒ `bbc` | `bbc` | 3 |
| 4 | `b c b a` (b ×2, c, a) ⇒ hết chuỗi | `bcba` | **4** |
| 5 | `c b a` | `cba` | 3 |
| 6 | `b a` | `ba` | 2 |
| 7 | `a` | `a` | 1 |

Đáp án lớn nhất là `4` (bắt đầu ở `4`). Để ý: càng đi sang phải, chuỗi con hợp lệ càng không thể "lùi lại" — đó là gợi ý cho cửa sổ trượt.

In [7]:
s = "bcbbbcba"
best = 0
for i in range(len(s)):
    j = i
    cnt = Counter()
    while j < len(s) and cnt[s[j]] < 2:       # còn có thể thêm s[j] mà không vượt 2 lần
        cnt[s[j]] += 1
        j += 1
    print(f"i={i}: dài nhất hợp lệ = {s[i:j]!r} (độ dài {j - i})")
    best = max(best, j - i)
print("đáp án:", best)

i=0: dài nhất hợp lệ = 'bcb' (độ dài 3)
i=1: dài nhất hợp lệ = 'cbb' (độ dài 3)
i=2: dài nhất hợp lệ = 'bb' (độ dài 2)
i=3: dài nhất hợp lệ = 'bbc' (độ dài 3)
i=4: dài nhất hợp lệ = 'bcba' (độ dài 4)
i=5: dài nhất hợp lệ = 'cba' (độ dài 3)
i=6: dài nhất hợp lệ = 'ba' (độ dài 2)
i=7: dài nhất hợp lệ = 'a' (độ dài 1)
đáp án: 4


## 5. Cách 1 — Brute Force / baseline: thử mọi chuỗi con và đếm lại

**Ý tưởng.** Với mọi cặp `i < j`, lấy `sub = s[i:j]`, đếm số lần mỗi ký tự bằng `Counter`, nếu `max(count) <= 2` thì cập nhật đáp án bằng `j - i`. Đơn giản và đúng, nhưng làm đi làm lại cùng một công việc đếm.

In [8]:
def max_length_bruteforce(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i + 1, n + 1):                 # chuỗi con s[i:j]
            sub = s[i:j]
            if max(Counter(sub).values()) <= 2:       # mọi ký tự xuất hiện không quá 2 lần
                best = max(best, j - i)
    return best


for s, expected in examples:
    print(repr(s), max_length_bruteforce(s), "(kỳ vọng", expected, ")")

'bcbbbcba' 4 (kỳ vọng 4 )
'aaaa' 2 (kỳ vọng 2 )
'abcabcabc' 6 (kỳ vọng 6 )


**Độ phức tạp (Cách 1).** Có `O(n²)` chuỗi con, mỗi chuỗi tốn `O(n)` để cắt và đếm ⇒ **Time `O(n³)`**, **Space `O(n)`**. Với `n <= 100` là `10^6` thao tác: chạy được, nhưng lãng phí.

**Nút thắt:** khi `sub = s[i:j]` đã hợp lệ, `s[i:j+1]` chỉ khác **một ký tự**; ta tính lại từ đầu. Và nếu `s[i:j]` đã vi phạm thì mọi `s[i:j']` với `j' > j` **cũng vi phạm**, nên không cần thử tiếp.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Tính đơn điệu theo `j`:** nếu `s[i:j]` vi phạm (có ký tự `> 2` lần) thì chuỗi dài hơn `s[i:j']` (`j' > j`) cũng vi phạm. Vậy với mỗi `i`, chỉ cần **mở rộng `j` đến khi vi phạm**, rồi dừng (Cách 2, `O(n²)`).
2. **Tính đơn điệu theo `i`:** nếu `s[i:j]` hợp lệ thì `s[i+1:j]` (bỏ bớt đầu) cũng hợp lệ. Do đó nếu `j` là điểm vi phạm đầu tiên với `i`, thì với `i + 1` điểm vi phạm đầu tiên **không nhỏ hơn** `j`. Nghĩa là **`j` không bao giờ phải lùi**: ta chỉ dịch `i` sang phải và `j` sang phải ⇒ **cửa sổ trượt** `O(n)`.

Thuật toán cửa sổ trượt: `right` thêm ký tự mới; **nếu ký tự đó vượt `2` lần** thì dịch `left` sang phải (giảm đếm) cho tới khi hết vi phạm; sau đó cập nhật đáp án với độ dài cửa sổ.

In [9]:
# Quan sát đơn điệu: điểm vi phạm đầu tiên j(i) không giảm khi i tăng
def first_violation(s, i):
    cnt = Counter()
    j = i
    while j < len(s) and cnt[s[j]] < 2:
        cnt[s[j]] += 1
        j += 1
    return j                      # s[i:j] hợp lệ, s[i:j+1] (nếu còn) vi phạm


s = "bcbbbcba"
js = [first_violation(s, i) for i in range(len(s))]
print(js)                                         # [3, 4, 4, 6, 8, 8, 8, 8]
assert all(js[k] <= js[k + 1] for k in range(len(js) - 1))
print("j không bao giờ giảm khi i tăng")

[3, 4, 4, 6, 8, 8, 8, 8]
j không bao giờ giảm khi i tăng


## 7. Các cách tiếp theo

### Cách 2 — Với mỗi điểm bắt đầu, mở rộng sang phải tới khi vi phạm

**Giải thích.** Với mỗi `i`, tạo bộ đếm mới, `j` chạy sang phải; dừng khi ký tự `s[j]` đã xuất hiện `2` lần (thêm nữa sẽ là `3`). Độ dài hợp lệ là `j - i`. Không cắt chuỗi, không đếm lại từ đầu mỗi lần.

In [10]:
def max_length_expand(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        cnt = {}
        j = i
        while j < n and cnt.get(s[j], 0) < 2:         # thêm s[j] vẫn không vượt 2 lần
            cnt[s[j]] = cnt.get(s[j], 0) + 1
            j += 1
        best = max(best, j - i)
    return best


for s, expected in examples:
    print(repr(s), max_length_expand(s), "(kỳ vọng", expected, ")")

'bcbbbcba' 4 (kỳ vọng 4 )
'aaaa' 2 (kỳ vọng 2 )
'abcabcabc' 6 (kỳ vọng 6 )


**Độ phức tạp Cách 2:** **Time `O(n²)`** (hai vòng lồng), **Space `O(1)`** (tối đa 26 ký tự trong `dict`).

### Cách 3 — Cửa sổ trượt với `Counter`

**Giải thích từng đoạn.**

1. `left = 0`, `cnt = Counter()`, `best = 0`.
2. Với mỗi `right`: thêm `cnt[s[right]] += 1`.
3. `while cnt[s[right]] > 2`: bỏ ký tự `s[left]` (giảm đếm), `left += 1`. (Chỉ ký tự **vừa thêm** mới có thể vượt `2`, vì trước đó cửa sổ hợp lệ.)
4. Sau vòng `while`, cửa sổ `s[left..right]` hợp lệ; `best = max(best, right - left + 1)`.

In [11]:
def max_length_window(s: str) -> int:
    cnt = Counter()
    left = 0
    best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1                                   # thêm ký tự mới vào cửa sổ
        while cnt[ch] > 2:                             # ch vượt 2 lần -> thu hẹp từ trái
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)             # cửa sổ lúc này hợp lệ
    return best


for s, expected in examples:
    print(repr(s), max_length_window(s), "(kỳ vọng", expected, ")")

'bcbbbcba' 4 (kỳ vọng 4 )
'aaaa' 2 (kỳ vọng 2 )
'abcabcabc' 6 (kỳ vọng 6 )


In [12]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionWindow:
    def maximumLengthSubstring(self, s: str) -> int:
        cnt = Counter()
        left = best = 0
        for right, ch in enumerate(s):
            cnt[ch] += 1
            while cnt[ch] > 2:
                cnt[s[left]] -= 1
                left += 1
            best = max(best, right - left + 1)
        return best


print(SolutionWindow().maximumLengthSubstring("bcbbbcba"))     # 4

4


**Độ phức tạp Cách 3:** `right` chạy `n` lần; `left` tổng cộng tăng tối đa `n` lần (mỗi chỉ số bị bỏ đúng một lần) ⇒ **Time `O(n)`**, **Space `O(1)`** (tối đa 26 khóa).

### Cách 4 — Cửa sổ trượt với mảng đếm 26

**Giải thích.** Cùng thuật toán, thay `Counter` bằng list 26 phần tử: nhanh hơn một chút và không phụ thuộc thư viện. Ta bỏ `Counter`, dùng `idx = ord(ch) - 97`.

In [13]:
def max_length_array(s: str) -> int:
    count = [0] * 26
    left = best = 0
    for right in range(len(s)):
        r = ord(s[right]) - 97
        count[r] += 1
        while count[r] > 2:
            count[ord(s[left]) - 97] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


for s, expected in examples:
    print(repr(s), max_length_array(s), "(kỳ vọng", expected, ")")

'bcbbbcba' 4 (kỳ vọng 4 )
'aaaa' 2 (kỳ vọng 2 )
'abcabcabc' 6 (kỳ vọng 6 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(26) = O(1)`**.

### Cách 5 — Nhớ hai vị trí gần nhất của mỗi ký tự, nhảy thẳng `left` (không cần `while`)

**Giải thích.** Thay vì bỏ từng ký tự bên trái, ta **tính luôn** `left` mới. Với mỗi ký tự, giữ hai chỉ số **xuất hiện gần nhất** `(cũ, mới)` (ban đầu `(-1, -1)`). Khi ký tự `ch` xuất hiện ở `right`:

- Nếu `ch` đã xuất hiện **ít nhất hai lần** trước đó (có `cũ != -1`) thì cửa sổ chứa cả ba lần xuất hiện là vi phạm; để hợp lệ, `left` phải **lớn hơn `cũ`**, tức `left = max(left, cũ + 1)`.
- Sau đó cập nhật cặp thành `(mới, right)`.
- Độ dài cửa sổ là `right - left + 1`.

Không còn vòng `while` bên trong; mỗi bước `O(1)`. (Nếu `cũ = -1` thì `cũ + 1 = 0` nên `max(left, 0)` không đổi `left`.)

Lưu ý: không nên "rút gọn" bằng cách thay `while` bằng `if` rồi dịch cả cửa sổ một bước — cách đó **sai** vì có thể cần bỏ nhiều ký tự (xem Lỗi 1 ở §13).

In [14]:
def max_length_positions(s: str) -> int:
    last = {}                                  # ch -> (vị trí cũ, vị trí mới) của hai lần xuất hiện gần nhất
    left = best = 0
    for right, ch in enumerate(s):
        older, newer = last.get(ch, (-1, -1))
        left = max(left, older + 1)            # nếu ch đã có 2 lần: cửa sổ phải bắt đầu sau lần cũ hơn
        last[ch] = (newer, right)              # trượt cặp vị trí
        best = max(best, right - left + 1)
    return best


for s, expected in examples:
    print(repr(s), max_length_positions(s), "(kỳ vọng", expected, ")")

'bcbbbcba' 4 (kỳ vọng 4 )
'aaaa' 2 (kỳ vọng 2 )
'abcabcabc' 6 (kỳ vọng 6 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (tối đa 26 khóa, mỗi khóa hai số).

Vì sao đúng? Với mỗi `right`, `left` bằng chỉ số lớn nhất trong `{older + 1}` của mọi ký tự đã xét. Với từng ký tự `c`, cửa sổ chứa không quá hai lần xuất hiện của `c` khi và chỉ khi `left` lớn hơn vị trí xuất hiện thứ ba tính ngược từ `right`; ta cập nhật `left` ngay khi gặp lần xuất hiện mới của `c`, nên `left` luôn là chỉ số nhỏ nhất làm mọi ký tự hợp lệ.

## 8. TRACE từng bước

Ta trace Cách 3 (cửa sổ co lại) trên `"bcbbbcba"` để thấy từng lần thu hẹp.

In [15]:
def trace_window(s: str) -> int:
    cnt = Counter()
    left = best = 0
    print("s =", s, "(chỉ số:", " ".join(str(i) for i in range(len(s))) + ")")
    for right, ch in enumerate(s):
        cnt[ch] += 1
        print(f"right={right} thêm {ch!r}: cnt={dict(cnt)}")
        while cnt[ch] > 2:
            dropped = s[left]
            cnt[dropped] -= 1
            left += 1
            print(f"    {ch!r} xuất hiện > 2 -> bỏ s[{left - 1}]={dropped!r}, left={left}, cnt={dict(cnt)}")
        best = max(best, right - left + 1)
        print(f"    cửa sổ s[{left}..{right}] = {s[left:right + 1]!r}, dài {right - left + 1}, best={best}")
    return best


assert trace_window("bcbbbcba") == 4

s = bcbbbcba (chỉ số: 0 1 2 3 4 5 6 7)
right=0 thêm 'b': cnt={'b': 1}
    cửa sổ s[0..0] = 'b', dài 1, best=1
right=1 thêm 'c': cnt={'b': 1, 'c': 1}
    cửa sổ s[0..1] = 'bc', dài 2, best=2
right=2 thêm 'b': cnt={'b': 2, 'c': 1}
    cửa sổ s[0..2] = 'bcb', dài 3, best=3
right=3 thêm 'b': cnt={'b': 3, 'c': 1}
    'b' xuất hiện > 2 -> bỏ s[0]='b', left=1, cnt={'b': 2, 'c': 1}
    cửa sổ s[1..3] = 'cbb', dài 3, best=3
right=4 thêm 'b': cnt={'b': 3, 'c': 1}
    'b' xuất hiện > 2 -> bỏ s[1]='c', left=2, cnt={'b': 3, 'c': 0}
    'b' xuất hiện > 2 -> bỏ s[2]='b', left=3, cnt={'b': 2, 'c': 0}
    cửa sổ s[3..4] = 'bb', dài 2, best=3
right=5 thêm 'c': cnt={'b': 2, 'c': 1}
    cửa sổ s[3..5] = 'bbc', dài 3, best=3
right=6 thêm 'b': cnt={'b': 3, 'c': 1}
    'b' xuất hiện > 2 -> bỏ s[3]='b', left=4, cnt={'b': 2, 'c': 1}
    cửa sổ s[4..6] = 'bcb', dài 3, best=3
right=7 thêm 'a': cnt={'b': 2, 'c': 1, 'a': 1}
    cửa sổ s[4..7] = 'bcba', dài 4, best=4


In [16]:
def trace_positions(s: str) -> int:
    last = {}
    left = best = 0
    for right, ch in enumerate(s):
        older, newer = last.get(ch, (-1, -1))
        new_left = max(left, older + 1)
        note = f"left {left} -> {new_left}" if new_left != left else "left giữ nguyên"
        left = new_left
        last[ch] = (newer, right)
        best = max(best, right - left + 1)
        print(f"right={right} ch={ch!r}: hai vị trí gần nhất trước đó=({older},{newer}) -> {note}; cửa sổ s[{left}..{right}] dài {right - left + 1}; best={best}")
    return best


assert trace_positions("bcbbbcba") == 4

right=0 ch='b': hai vị trí gần nhất trước đó=(-1,-1) -> left giữ nguyên; cửa sổ s[0..0] dài 1; best=1
right=1 ch='c': hai vị trí gần nhất trước đó=(-1,-1) -> left giữ nguyên; cửa sổ s[0..1] dài 2; best=2
right=2 ch='b': hai vị trí gần nhất trước đó=(-1,0) -> left giữ nguyên; cửa sổ s[0..2] dài 3; best=3
right=3 ch='b': hai vị trí gần nhất trước đó=(0,2) -> left 0 -> 1; cửa sổ s[1..3] dài 3; best=3
right=4 ch='b': hai vị trí gần nhất trước đó=(2,3) -> left 1 -> 3; cửa sổ s[3..4] dài 2; best=3
right=5 ch='c': hai vị trí gần nhất trước đó=(-1,1) -> left giữ nguyên; cửa sổ s[3..5] dài 3; best=3
right=6 ch='b': hai vị trí gần nhất trước đó=(3,4) -> left 3 -> 4; cửa sổ s[4..6] dài 3; best=3
right=7 ch='a': hai vị trí gần nhất trước đó=(-1,-1) -> left giữ nguyên; cửa sổ s[4..7] dài 4; best=4


Bảng trace Cách 3 với `"bcbbbcba"` (chỉ ghi các bước quan trọng):

| right | thêm | cnt sau thêm | cần thu hẹp? | left | cửa sổ | best |
|---|---|---|---|---|---|---|
| 0 | b | {b:1} | không | 0 | `b` | 1 |
| 1 | c | {b:1, c:1} | không | 0 | `bc` | 2 |
| 2 | b | {b:2, c:1} | không | 0 | `bcb` | 3 |
| 3 | b | {b:3, c:1} | **có**: bỏ `s[0]=b` | 1 | `cbb` | 3 |
| 4 | b | {b:3, c:1} | **có**: bỏ `s[1]=c`, rồi `s[2]=b` | 3 | `bb` | 3 |
| 5 | c | {b:2, c:1} | không | 3 | `bbc` | 3 |
| 6 | b | {b:3, c:1} | **có**: bỏ `s[3]=b` | 4 | `bcb` | 3 |
| 7 | a | {b:2, c:1, a:1} | không | 4 | `bcba` | **4** |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (cửa sổ trượt với `Counter`): `O(n)`, dễ đọc, đúng và an toàn.

In [17]:
from collections import Counter

class Solution:
    def maximumLengthSubstring(self, s: str) -> int:
        cnt = Counter()
        left = 0
        best = 0
        for right, ch in enumerate(s):
            cnt[ch] += 1                         # thêm s[right] vào cửa sổ
            while cnt[ch] > 2:                   # ký tự này xuất hiện quá 2 lần
                cnt[s[left]] -= 1                # bỏ ký tự ngoài cùng bên trái
                left += 1
            best = max(best, right - left + 1)   # cửa sổ hợp lệ: cập nhật đáp án
        return best


sol = Solution()
for s, expected in examples:
    print(repr(s), "->", sol.maximumLengthSubstring(s))

'bcbbbcba' -> 4
'aaaa' -> 2
'abcabcabc' -> 6


## 10. Vì sao thuật toán đúng

- **Bất biến:** trước mỗi lần thêm `s[right]`, cửa sổ `s[left..right-1]` hợp lệ (mọi ký tự `<= 2` lần). Sau khi thêm, chỉ `s[right]` có thể vượt `2` (đếm đúng `3`); `while` bỏ các ký tự bên trái cho tới khi `s[right]` còn `2` ⇒ cửa sổ lại hợp lệ. Vì chỉ một ký tự có thể vượt, ta chỉ cần kiểm tra ký tự đó.
- **Không bỏ sót đáp án:** với mỗi `right`, `left` là **chỉ số nhỏ nhất** sao cho `s[left..right]` hợp lệ (nếu bỏ ít hơn thì `s[right]` vẫn vượt `2`). Cửa sổ dài nhất kết thúc tại `right` chính là `s[left..right]`; lấy `max` qua mọi `right` được đáp án.
- **`left` không phải lùi:** vì nếu `s[l..r]` hợp lệ thì `s[l'..r]` với `l' > l` cũng hợp lệ (tính đơn điệu), nên vị trí `left` nhỏ nhất hợp lệ không giảm khi `right` tăng.
- **Thời gian tuyến tính:** mỗi chỉ số vào cửa sổ đúng một lần (`right`) và ra tối đa một lần (`left`).

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi chuỗi con, đếm lại | `O(n³)` | `O(n)` |
| 2. Mở rộng từ mỗi điểm bắt đầu | `O(n²)` | `O(1)` |
| 3. Cửa sổ trượt `Counter` | `O(n)` | `O(1)` |
| 4. Cửa sổ trượt mảng đếm 26 | `O(n)` | `O(1)` |
| 5. Nhớ 2 vị trí gần nhất, nhảy `left` | `O(n)` | `O(1)` |

(Không gian `O(1)` vì bảng chữ cái cố định `26`.)

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "brute": max_length_bruteforce,
    "expand": max_length_expand,
    "window": max_length_window,
    "array": max_length_array,
    "positions": max_length_positions,
    "leetcode": sol.maximumLengthSubstring,
}


def check_all(s, expected, label):
    for key, fn in impls.items():
        actual = fn(s)
        assert actual == expected, (label, key, s, actual, expected)
    print("PASS", label)


for idx, (s, expected) in enumerate(examples, 1):
    check_all(s, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("aa", 2, "độ dài tối thiểu theo đề, hai ký tự giống nhau")
check_all("ab", 2, "hai ký tự khác nhau")
check_all("aaa", 2, "ba ký tự giống nhau")
check_all("aab", 3, "a hai lần, b một lần")
check_all("aaab", 3, "aab ở cuối")
check_all("abcdefghijklmnopqrstuvwxyz", 26, "26 ký tự khác nhau")
check_all("abcdefghijklmnopqrstuvwxyz" * 2, 52, "mỗi chữ hai lần")
check_all("abcdefghijklmnopqrstuvwxyz" * 3 + "ab", 52, "ba vòng bảng chữ cái: cửa sổ tối đa 52")
check_all("a" * 100, 2, "n = 100 toàn chữ a")
check_all("ab" * 50, 4, "abab...: abab dài 4 (a, b mỗi chữ hai lần)")
check_all("aabbaabb", 4, "aabb, bbaa, aabb...: mỗi nhóm 4")

PASS độ dài tối thiểu theo đề, hai ký tự giống nhau
PASS hai ký tự khác nhau
PASS ba ký tự giống nhau
PASS a hai lần, b một lần
PASS aab ở cuối
PASS 26 ký tự khác nhau
PASS mỗi chữ hai lần
PASS ba vòng bảng chữ cái: cửa sổ tối đa 52
PASS n = 100 toàn chữ a
PASS abab...: abab dài 4 (a, b mỗi chữ hai lần)
PASS aabb, bbaa, aabb...: mỗi nhóm 4


### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **dùng `if` thay `while`** khi thu hẹp trong cửa sổ co lại (ghi đáp án vẫn như cũ); (b) **cập nhật đáp án trước khi thu hẹp**. Ta tạo hàm sai rồi tìm input phân biệt.

In [20]:
def wrong_if_instead_of_while(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        if cnt[ch] > 2:                          # SAI: chỉ bỏ một ký tự bên trái
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)       # ghi đáp án cả khi cửa sổ có thể chưa hợp lệ
    return best


def wrong_update_before_shrink(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        best = max(best, right - left + 1)       # SAI: ghi trước khi thu hẹp
        while cnt[ch] > 2:
            cnt[s[left]] -= 1
            left += 1
    return best


bad1 = "bcbbbcba"
bad2 = "aaaa"
print(wrong_if_instead_of_while(bad1), "!=", max_length_window(bad1))
print(wrong_update_before_shrink(bad2), "!=", max_length_window(bad2))
assert wrong_if_instead_of_while(bad1) != max_length_window(bad1)
assert wrong_update_before_shrink(bad2) != max_length_window(bad2)

5 != 4
3 != 2


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh chuỗi đúng ràng buộc (độ dài `2..100`, chữ thường), với bảng chữ cái nhỏ để buộc cửa sổ co lại nhiều lần. Tham chiếu: Cách 1 (đúng theo định nghĩa) với độ dài `<= 40`, các cách nhanh với nhau cho độ dài lớn.

In [21]:
rng = random.Random(3090)
for trial in range(1500):
    alphabet = rng.choice(["a", "ab", "abc", "abcd", "abcdefg"])
    n = rng.randint(2, 40)
    s = "".join(rng.choice(alphabet) for _ in range(n))
    ref = max_length_bruteforce(s)
    for key, fn in impls.items():
        assert fn(s) == ref, (key, s)
print("PASS 1500 chuỗi nhỏ khớp brute force theo định nghĩa")

for trial in range(2500):
    alphabet = rng.choice(["a", "ab", "abc", "abcdefghij", string.ascii_lowercase])
    n = rng.choice([2, 10, 50, 99, 100])
    s = "".join(rng.choice(alphabet) for _ in range(n))
    assert 2 <= len(s) <= 100 and s.islower()
    ref = max_length_expand(s)
    for key, fn in impls.items():
        if key == "brute":
            continue
        assert fn(s) == ref, (key, s)
print("PASS 2500 chuỗi lớn: các cách nhanh khớp nhau (kể cả cách nhảy `left`)")

PASS 1500 chuỗi nhỏ khớp brute force theo định nghĩa


PASS 2500 chuỗi lớn: các cách nhanh khớp nhau (kể cả cách nhảy `left`)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `if` thay `while` khi thu hẹp cửa sổ co lại

In [22]:
def sai_dung_if(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        if cnt[ch] > 2:                          # SAI: có thể phải bỏ nhiều ký tự
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


print("sai:", sai_dung_if("bcbbbcba"), "| đúng:", Solution().maximumLengthSubstring("bcbbbcba"))

sai: 5 | đúng: 4


**Vì sao sai:** để bỏ bớt một ký tự `b` ta có thể phải bỏ **nhiều** ký tự bên trái (ở bước `right = 4` ta phải bỏ `c` rồi tới `b`, tức hai ký tự). `if` chỉ bỏ một ký tự nên cửa sổ vẫn vi phạm, nhưng ta vẫn ghi `best` ⇒ có thể quá lớn. **Cách sửa:** `while cnt[ch] > 2` (hoặc nhảy thẳng `left` như Cách 5).

### Lỗi 2: Quên giảm đếm khi bỏ ký tự bên trái

In [23]:
def sai_quen_giam_dem(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > 2:
            left += 1                            # SAI: không giảm cnt của s[left]
        best = max(best, right - left + 1)
    return best


try:
    import signal
    def handler(signum, frame):
        raise TimeoutError("vòng lặp vô hạn hoặc quá lâu")
    signal.signal(signal.SIGALRM, handler)
    signal.alarm(1)
    print(sai_quen_giam_dem("aaa"))
except TimeoutError as e:
    print("sai ->", e)
except Exception as e:
    print("sai ->", type(e).__name__, e)
finally:
    signal.alarm(0)
print("đúng:", Solution().maximumLengthSubstring("aaa"))

sai -> vòng lặp vô hạn hoặc quá lâu
đúng: 2


**Vì sao sai:** nếu không giảm `cnt`, điều kiện `cnt[ch] > 2` **không bao giờ hết đúng**, vòng `while` lặp vô hạn (ở đây `left` tăng mãi, vượt cả độ dài chuỗi). **Cách sửa:** `cnt[s[left]] -= 1` trước `left += 1`.

### Lỗi 3: Hiểu "tối đa 2 lần" thành "các ký tự khác nhau" (bài Longest Substring Without Repeating Characters)

In [24]:
def sai_khong_lap(s):
    seen = set()
    left = best = 0
    for right, ch in enumerate(s):
        while ch in seen:                        # SAI: coi lần lặp thứ 2 là vi phạm
            seen.remove(s[left])
            left += 1
        seen.add(ch)
        best = max(best, right - left + 1)
    return best


print("sai:", sai_khong_lap("aaab"), "| đúng:", Solution().maximumLengthSubstring("aaab"))

sai: 2 | đúng: 3


**Vì sao sai:** đề cho phép **mỗi ký tự xuất hiện tối đa `2` lần**, không phải `1`. Dùng `set` chỉ nhớ "có / không" chứ không đếm. **Cách sửa:** dùng bộ đếm (`Counter` / mảng 26) và so với `2`.

### Lỗi 4: Cắt chuỗi con rồi đếm trong vòng lặp (chậm, nhưng dễ nhầm biên)

In [25]:
def sai_bien(s):
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i + 1, n):                # SAI: j chỉ tới n - 1, bỏ sót chuỗi con kết thúc ở cuối
            sub = s[i:j]
            if max(Counter(sub).values()) <= 2:
                best = max(best, j - i)
    return best


print("sai:", sai_bien("ab"), "| đúng:", Solution().maximumLengthSubstring("ab"))

sai: 1 | đúng: 2


**Vì sao sai:** `s[i:j]` **không gồm** `s[j]`; để lấy tới ký tự cuối, `j` phải chạy tới `n` (`range(i + 1, n + 1)`). Sai biên cho `"ab"` ra `1` thay vì `2`. **Cách sửa:** `for j in range(i + 1, n + 1)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**Chuỗi con / mảng con liên tiếp dài nhất** thỏa một điều kiện đếm" ⇒ nghĩ ngay **cửa sổ trượt**. Điều kiện cần: nếu cửa sổ hợp lệ thì **bỏ bớt** từ hai đầu vẫn hợp lệ (tính đơn điệu).
- Khung mẫu: `for right in ...: thêm; while vi_phạm: bỏ trái; cập_nhật best`.
- Chỉ **ký tự vừa thêm** mới có thể làm vi phạm, nên kiểm tra riêng nó (không cần quét toàn bộ bộ đếm).
- Đổi `2` thành `k` ⇒ tổng quát hóa (Cách `at most k occurrences`). Đổi `2` thành `1` ⇒ *Longest Substring Without Repeating Characters (3)*.
- Bảng chữ cái nhỏ ⇒ mảng đếm 26; điều kiện `<= 2` đơn giản.

## 15. Biến thể của bài

1. **Mỗi ký tự tối đa `k` lần**: thay `2` bằng tham số `k`.
2. **Trả về chính chuỗi con** (hoặc vị trí đầu và cuối) thay vì độ dài.
3. **Longest Substring Without Repeating Characters (3)**: `k = 1`.
4. **Longest Substring with At Most K Distinct Characters (340)**: giới hạn số ký tự **khác nhau**, không phải số lần.
5. **Max Consecutive Ones III (1004)**, **Fruit Into Baskets (904)**: cửa sổ trượt cùng khung.

In [26]:
def max_length_k(s, k):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > k:
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


def longest_substring_text(s, k=2):
    cnt = Counter()
    left = 0
    best_len, best_start = 0, 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > k:
            cnt[s[left]] -= 1
            left += 1
        if right - left + 1 > best_len:
            best_len, best_start = right - left + 1, left
    return s[best_start:best_start + best_len]


print(max_length_k("bcbbbcba", 1), max_length_k("bcbbbcba", 2), max_length_k("bcbbbcba", 3))   # 3 4 6
print(longest_substring_text("bcbbbcba"))
assert max_length_k("bcbbbcba", 2) == 4
assert max_length_k("abca", 1) == 3
assert longest_substring_text("aaab") == "aab"
assert len(longest_substring_text("bcbbbcba")) == 4

3 4 5
bcba


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi chuỗi con, đếm lại | đúng định nghĩa | `O(n³)` | `O(n)` | kiểm chứng, `n` rất nhỏ |
| 2. Mở rộng từ mỗi điểm đầu | dừng khi vi phạm | `O(n²)` | `O(1)` | khi chưa nhìn ra cửa sổ trượt |
| 3. Cửa sổ trượt `Counter` | `while` thu hẹp | `O(n)` | `O(1)` | **nên nộp**, dễ chứng minh |
| 4. Cửa sổ trượt mảng 26 | như trên, mảng thay dict | `O(n)` | `O(1)` | cần tốc độ nhỏ nhất |
| 5. Nhớ 2 vị trí gần nhất | nhảy thẳng `left = max(left, cũ + 1)` | `O(n)` | `O(1)` | khi muốn tránh `while` lồng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"abcd"[1:3]`, `Counter("aab")`, `max(Counter("aaab").values())`, `Solution().maximumLengthSubstring("aabb")`, `Solution().maximumLengthSubstring("aaabbb")`.

**Bài 2 (tính tay).** Với `s = "abaabab"`: lập bảng cửa sổ trượt (mỗi `right`: `cnt`, `left`, cửa sổ, `best`) và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_length_todo(s)` theo Cách 3 (cửa sổ trượt `Counter`).

In [27]:
def max_length_todo(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        # TODO 1: tăng cnt[ch] lên 1
        # TODO 2: while cnt[ch] > 2: giảm cnt[s[left]] đi 1 và tăng left lên 1
        # TODO 3: cập nhật best = max(best, độ dài cửa sổ)
        pass
    return best

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_length_expand2(s)` theo Cách 2 (mở rộng từ mỗi điểm bắt đầu) **không dùng `Counter`**, chỉ dùng list 26 phần tử.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_at_most_k_distinct(s, k)`: độ dài chuỗi con dài nhất có **tối đa `k` ký tự khác nhau**. (b) Viết `count_valid_substrings(s)`: **số chuỗi con** (tính theo vị trí) mà mỗi ký tự xuất hiện tối đa 2 lần (gợi ý: với mỗi `right`, số chuỗi con hợp lệ kết thúc tại `right` là `right - left + 1`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `"bc"`, `Counter({'a': 2, 'b': 1})`, `3`, `4`, `4` (với `"aaabbb"` là `"aabb"`). Bài 3: ba chỗ TODO đúng ba câu lệnh.
- **Mức 2 (vừa):** Bài 2: các `right` lần lượt `0..6`; bước đáng chú ý là khi thêm `a` thứ ba (ở `right = 3`, `s = a b a a ...`): bỏ `s[0] = a`. Đáp án `4`. Bài 4: mỗi `i` dùng list 26 mới.
- **Mức 3 (sâu):** Bài 5(a): thêm biến `distinct` đếm số ký tự có đếm `> 0`; thu hẹp khi `distinct > k`. 5(b): cộng `right - left + 1` vào `total` sau khi thu hẹp; kiểm chứng bằng brute force cho chuỗi ngắn.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "abcd"[1:3] == "bc"
assert Counter("aab") == Counter({"a": 2, "b": 1})
assert max(Counter("aaab").values()) == 3
assert Solution().maximumLengthSubstring("aabb") == 4
assert Solution().maximumLengthSubstring("aaabbb") == 4

# Bài 2
assert Solution().maximumLengthSubstring("abaabab") == 4      # abaab? a x3; "baab"/"abab" dài 4


# Bài 3
def max_length_todo_ans(s):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > 2:
            cnt[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best


for s, expected in examples:
    assert max_length_todo_ans(s) == expected


# Bài 4
def max_length_expand2(s):
    n = len(s)
    best = 0
    for i in range(n):
        count = [0] * 26
        j = i
        while j < n and count[ord(s[j]) - 97] < 2:
            count[ord(s[j]) - 97] += 1
            j += 1
        best = max(best, j - i)
    return best


for s, expected in examples:
    assert max_length_expand2(s) == expected


# Bài 5(a)
def longest_at_most_k_distinct(s, k):
    cnt = Counter()
    left = best = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while len(cnt) > k:
            cnt[s[left]] -= 1
            if cnt[s[left]] == 0:
                del cnt[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best


assert longest_at_most_k_distinct("eceba", 2) == 3
assert longest_at_most_k_distinct("aa", 1) == 2
assert longest_at_most_k_distinct("abc", 1) == 1


# Bài 5(b)
def count_valid_substrings(s):
    cnt = Counter()
    left = total = 0
    for right, ch in enumerate(s):
        cnt[ch] += 1
        while cnt[ch] > 2:
            cnt[s[left]] -= 1
            left += 1
        total += right - left + 1
    return total


def count_valid_bruteforce(s):
    return sum(max(Counter(s[i:j]).values()) <= 2 for i in range(len(s)) for j in range(i + 1, len(s) + 1))


assert count_valid_substrings("aaa") == 5            # a, a, a, aa, aa
rng = random.Random(21)
for _ in range(300):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 14)))
    assert count_valid_substrings(s) == count_valid_bruteforce(s)
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cửa sổ trượt: mỗi ký tự tối đa 2 lần)

```python
from collections import Counter

class Solution:
    def maximumLengthSubstring(self, s: str) -> int:
        cnt = Counter()
        left = best = 0
        for right, ch in enumerate(s):
            cnt[ch] += 1
            while cnt[ch] > 2:
                cnt[s[left]] -= 1
                left += 1
            best = max(best, right - left + 1)
        return best
```

**Checklist**

- [ ] Dùng `while` (không phải `if`) để thu hẹp, và **giảm đếm** của `s[left]` trước khi `left += 1`?
- [ ] Cập nhật `best` **sau** khi thu hẹp (cửa sổ đã hợp lệ)?
- [ ] Độ dài cửa sổ là `right - left + 1` (cả hai đầu gồm)?
- [ ] Điều kiện là `> 2` (cho phép đúng 2 lần), không phải `> 1` hay "đã thấy"?
- [ ] Đã thử `"aaaa"`, `"aa"` (n nhỏ nhất) và chuỗi toàn ký tự khác nhau?
- [ ] Skeleton đúng `class Solution` / `def maximumLengthSubstring(self, s: str) -> int`?